# 01 · Exploración de datos (EDA) y preprocesamiento
**Equipo 4 · Bank Marketing Dataset (UCI)**

## 1. Descripción del problema
Un banco realiza campañas telefónicas para ofrecer un depósito a plazo. Queremos entender qué
clientes aceptan (`y = yes`) y qué características del cliente y de la campaña se relacionan con ello.

**Preguntas del EDA**
1. ¿Qué tan desbalanceada está la variable objetivo?
2. ¿Qué variables numéricas y categóricas se relacionan con la aceptación?
3. ¿Cómo influyen las campañas de contacto y las variables temporales (mes, día)?
4. ¿Qué problemas de calidad hay (`unknown`, `pdays = -1`, colas largas, duplicados)?

> **Importante:** el EDA se hace solo con el conjunto de **entrenamiento** para que las decisiones
> de preprocesamiento no usen información del conjunto de prueba.

In [ ]:
import sys
sys.path.append("..")   # permite `from src import ...` al ejecutar desde notebooks/

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config
from src.data_loader import get_train_test, load_data

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)
config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)

X_train, X_test, y_train, y_test = get_train_test()
print("Train:", X_train.shape, "| Test:", X_test.shape)

## 2. Comprensión de los datos

In [ ]:
df = X_train.assign(y=y_train)
display(df.head())
print(df.dtypes)
print("\nValores nulos por columna:\n", df.isna().sum()[df.isna().sum() > 0])
print("Filas duplicadas:", df.duplicated().sum())

In [ ]:
# 'unknown' es un valor faltante disfrazado en las categóricas
cat_cols = X_train.select_dtypes(exclude=np.number).columns.tolist()
num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
unknown = (X_train[cat_cols] == "unknown").mean().sort_values(ascending=False)
display((unknown[unknown > 0] * 100).round(1).to_frame("% unknown"))

**Hallazgos (completar):**
- Tamaño, tipos de datos y calidad general: …
- Variables con mayor proporción de `unknown` y qué hacer con ellas: …

## 3. Variable objetivo y desbalance

In [ ]:
conteo = y_train.value_counts().rename({0: "no", 1: "yes"})
print(conteo, "\n", (conteo / conteo.sum()).round(3))
ax = conteo.plot(kind="bar", color=["#4C72B0", "#DD8452"], rot=0)
ax.set_title("Distribución de la variable objetivo (train)"); ax.set_ylabel("clientes")
plt.savefig(config.FIGURES_DIR / "eda_objetivo.png", dpi=150, bbox_inches="tight"); plt.show()

**Hallazgos (completar):** proporción de "yes", por qué `accuracy` no sirve como única métrica y qué
estrategias usaremos (pesos de clase, métricas F1/Recall/Precision/ROC-AUC/PR-AUC): …

## 4. Variables numéricas

In [ ]:
display(X_train[num_cols].describe().T.round(2))
fig, axes = plt.subplots(2, 4, figsize=(18, 7))
for ax, col in zip(axes.ravel(), num_cols):
    p99 = df[col].quantile(0.99)   # solo para graficar: los extremos aplastan la distribución
    sns.histplot(data=df[df[col] <= p99], x=col, hue="y", bins=40, ax=ax, element="step", stat="density", common_norm=False)
    ax.set_title(f"{col} (hasta p99 = {p99:,.0f})")
for ax in axes.ravel()[len(num_cols):]:
    ax.axis("off")
plt.tight_layout(); plt.savefig(config.FIGURES_DIR / "eda_numericas.png", dpi=150); plt.show()

In [ ]:
# Valores atípicos (regla del rango intercuartílico)
q1, q3 = X_train[num_cols].quantile(0.25), X_train[num_cols].quantile(0.75)
iqr = q3 - q1
outliers = ((X_train[num_cols] < q1 - 1.5 * iqr) | (X_train[num_cols] > q3 + 1.5 * iqr)).mean()
display((outliers.sort_values(ascending=False) * 100).round(1).to_frame("% atípicos"))

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(X_train[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlación entre variables numéricas")
plt.savefig(config.FIGURES_DIR / "eda_correlacion.png", dpi=150, bbox_inches="tight"); plt.show()

**Hallazgos (completar):** distribuciones asimétricas (`balance`, `campaign`, `previous`), atípicos,
correlaciones relevantes y qué transformación justifican (log, escalado): …

## 5. Variables categóricas (tasa de aceptación por categoría)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(18, 13))
for ax, col in zip(axes.ravel(), [c for c in cat_cols if c != "month"]):
    rate = df.groupby(col)["y"].mean().sort_values()
    rate.plot(kind="barh", ax=ax, color="#4C72B0")
    ax.axvline(df["y"].mean(), color="red", ls="--", lw=1)
    ax.set_title(f"Tasa de aceptación por {col}"); ax.set_xlabel("")
for ax in axes.ravel()[len(cat_cols) - 1:]:
    ax.axis("off")
plt.tight_layout(); plt.savefig(config.FIGURES_DIR / "eda_categoricas.png", dpi=150); plt.show()

**Hallazgos (completar):** categorías con tasa de aceptación muy por encima o por debajo del promedio (línea roja): …

## 6. Campañas de contacto y variables temporales

In [ ]:
df["previously_contacted"] = (df["pdays"] != -1)
print("Clientes contactados antes:", f"{df['previously_contacted'].mean():.1%}")
display(df.groupby("previously_contacted")["y"].agg(clientes="size", tasa_aceptacion="mean"))
display(df.groupby("poutcome")["y"].agg(clientes="size", tasa_aceptacion="mean"))

df["campaign_bin"] = pd.cut(df["campaign"], [0, 1, 2, 3, 5, 10, 100], labels=["1", "2", "3", "4-5", "6-10", ">10"])
display(df.groupby("campaign_bin", observed=True)["y"].agg(clientes="size", tasa_aceptacion="mean"))

In [ ]:
order = ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"]
by_month = df.groupby("month")["y"].agg(clientes="size", tasa="mean").reindex(order)
fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.bar(by_month.index, by_month["clientes"], color="#C9D6EA"); ax1.set_ylabel("contactos")
ax2 = ax1.twinx(); ax2.plot(by_month.index, by_month["tasa"], color="#DD8452", marker="o"); ax2.set_ylabel("tasa de aceptación")
plt.title("Volumen de contactos y aceptación por mes")
plt.savefig(config.FIGURES_DIR / "eda_mes.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# Justificación de excluir 'duration': solo se conoce tras terminar la llamada
raw = load_data()
display(raw.groupby("y")["duration"].describe().round(1))

**Hallazgos (completar):** efecto de contactos previos y `poutcome`, número de contactos de la campaña,
estacionalidad por mes y por qué `duration` se excluye (fuga de información): …

## 7. Preprocesamiento
Pipeline definido en `src/preprocessing.py` (se ajusta solo con entrenamiento):

| Paso | Decisión | Justificación |
|---|---|---|
| `pdays = -1` | Bandera `previously_contacted` y `pdays` → NaN | -1 no es una distancia real |
| `balance`, `campaign`, `previous` | Log con signo | Colas largas |
| `month` | Seno/coseno del mes | Variable cíclica |
| Numéricas | Mediana + `StandardScaler` | Robustez a nulos y escala comparable |
| Categóricas | `unknown` como categoría + One-Hot | `unknown` puede ser informativo |
| `duration` | Excluida | Fuga de información |

In [ ]:
from src.preprocessing import build_preprocessor, get_feature_names

prep = build_preprocessor().fit(X_train)          # fit SOLO con train
Xt_train, Xt_test = prep.transform(X_train), prep.transform(X_test)
print("Matriz train:", Xt_train.shape, "| test:", Xt_test.shape)
print("NaN restantes:", np.isnan(Xt_train).sum())
print("Primeras columnas:", get_feature_names(prep)[:12])

**Decisiones y limitaciones (completar):** ¿qué otra transformación probarían (agrupar categorías raras,
selección de características)? ¿Qué riesgo tiene tratar `unknown` como categoría?: …

## 8. Selección de variables y reducción de dimensionalidad (exploratoria)
Antes de modelar, medimos **qué variables aportan información sobre `y`** (información mutua, calculada solo
con entrenamiento) y **cuánta redundancia hay** (varianza acumulada de PCA). La comparación formal de estrategias
de selección con modelos está en `02_supervisado.ipynb`.

In [ ]:
from sklearn.feature_selection import mutual_info_classif
from sklearn.decomposition import PCA

names = get_feature_names(prep)
mi = pd.Series(mutual_info_classif(Xt_train, y_train, random_state=config.RANDOM_STATE), index=names)
mi = mi.sort_values(ascending=False)
display(mi.head(15).round(4).to_frame("info_mutua"))

plt.figure(figsize=(7, 6)); mi.head(20)[::-1].plot(kind="barh", color="#4C72B0")
plt.title("Top 20 variables por información mutua con y (train)")
plt.savefig(config.FIGURES_DIR / "eda_info_mutua.png", dpi=150, bbox_inches="tight"); plt.show()
print("Variables con información mutua ≈ 0 (<0.001):", int((mi < 0.001).sum()), "de", len(mi))

In [ ]:
pca_full = PCA(random_state=config.RANDOM_STATE).fit(Xt_train)
cum = np.cumsum(pca_full.explained_variance_ratio_)
plt.figure(figsize=(7, 4)); plt.plot(range(1, len(cum) + 1), cum, marker="o", ms=3)
for thr in (0.90, 0.95):
    k = int(np.argmax(cum >= thr)) + 1
    plt.axhline(thr, ls="--", color="gray", lw=0.8); plt.annotate(f"{thr:.0%}: {k} comp.", (k, thr), textcoords="offset points", xytext=(8, -14))
plt.xlabel("Componentes principales"); plt.ylabel("Varianza acumulada"); plt.title("PCA sobre las variables preprocesadas")
plt.savefig(config.FIGURES_DIR / "eda_pca_varianza.png", dpi=150, bbox_inches="tight"); plt.show()
print(f"{Xt_train.shape[1]} variables tras el preprocesamiento → {int(np.argmax(cum >= 0.95)) + 1} componentes explican el 95 % de la varianza")

**Hallazgos (completar):** variables más informativas, variables prácticamente irrelevantes (candidatas a
eliminar), redundancia que revela el PCA (muchas columnas one-hot) y qué estrategia de selección recomiendan
probar en el componente supervisado: …